In [1]:
import os, sys
sys.path.append(os.path.abspath(os.path.join(os.curdir, '..')))
from src.utils import *
from src.iiwa_program import Iiwa14IKProgram, iiwa_limits_lower, iiwa_limits_upper
from pydrake.all import (
    StartMeshcat,
    Quaternion,
    RigidTransform,
    MinimumDistanceLowerBoundConstraint,
    SceneGraphInspector,
)
from src.generic_program import ProgramOptions

ikflow/config.py | Using device: 'cuda:0'


In [2]:
meshcat = StartMeshcat()
diagram = BuildEnv(meshcat=meshcat, directives_file = os.path.join(RepoDir(), "models/iiwa14/iiwa14_collision.yaml"))

program_options = ProgramOptions(
    # visualize=True,
    # joint_centering_cost=1e-6
)


program = Iiwa14IKProgram(diagram, options = program_options)
target_pose = np.array([0.29252776760806476, 0.4714304570247682, 0.8799253594708412, 0.37517094428362757, -0.6461691081943264, 0.40286211881371536, -0.5285965942054528], dtype=np.float32)
program.create_prog(target_pose, q_nominal = program.plant.GetDefaultPositions())

INFO:drake:Meshcat listening for connections at http://localhost:7001


WorldModel::LoadRobot: /home/tangles/.cache/jrl/urdfs/iiwa14_formatted_link_filepaths_absolute.urdf
URDFParser: Link size: 11
URDFParser: Joint size: 11
URDFParser: Done loading robot file /home/tangles/.cache/jrl/urdfs/iiwa14_formatted_link_filepaths_absolute.urdf
Initialized robot collision data structures in time 0.228683
/home/tangles/Urop/ikflow/ik-net-optimization


In [14]:
found = False
while not found:
    q = np.random.uniform(low=iiwa_limits_lower, 
                                    high = iiwa_limits_upper)
    program.plant.SetPositions(program.plant_context, q)
    if program.collision_free_constraint.eval_func(q = q) < 0.1:
        if program.frame.CalcPoseInWorld(program.plant_context).translation()[2] < 0.8:
            found = True
program.diagram.ForcedPublish(program.diagram_context)
pose = program.frame.CalcPoseInWorld(program.plant_context)


In [15]:
program.create_prog(np.array([*pose.translation(), *pose.rotation().ToQuaternion().wxyz()]))

In [16]:
pose = program.target_pose
pose = RigidTransform(Quaternion(pose[3], pose[4], pose[5], pose[6]), [pose[0], pose[1], pose[2]])
DrawAxes(pose, meshcat)

result = program.Solve()

print(result.is_success())

vars = result.get_x_val()
print(vars)

True
[-7.66585952e-01  1.07697906e-01  7.01176019e-01  2.01173618e+00
 -5.90726146e-01 -1.48155714e+00  3.22449130e-03 -2.75269935e-04
 -3.79238831e-04 -1.39473999e-02 -7.35897098e-04 -9.50823669e-04
  1.37139880e-03 -2.13258183e-04 -3.14536162e-03 -9.15621712e-03
  3.61418069e-03  8.13862954e-03 -4.13373002e-03  7.29008594e-03
  1.18874604e-02]


In [ ]:
import time
start = time.time()
q_sol = program.VarsToQ(vars)
print(time.time()-start)
program.plant.SetPositions(program.plant_context, q_sol)
program.diagram.ForcedPublish(program.diagram_context)





0.02913069725036621


In [ ]:
program.plant.SetPositions(program.plant_context, q)
program.diagram.ForcedPublish(program.diagram_context)